In [1]:
# %%
# نوتبوک: 05_test_option_verifier.ipynb
# هدف: تست کامل option_verifier_tool و retriever_tool

# %% [markdown]
# # تست Option Verifier Tool و Retriever Tool
# 
# این نوتبوک ابزارها را تست می‌کند:
# 1. تست TOON parser
# 2. تست option_verifier_tool با نمونه‌های ساده
# 3. تست با سوالات واقعی حقوقی
# 4. تست retriever_tool با فیلترها

# %% [markdown]
## 1️⃣ Setup و Imports

# %%
import os
import sys
from pathlib import Path

# اضافه کردن مسیر src به path
project_root = Path.cwd().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"✓ Project root: {project_root}")
print(f"✓ Src path: {src_path}")

# %%
# بارگذاری environment variables
from dotenv import load_dotenv
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print(f"✓ OPENROUTER API Key loaded: {OPENROUTER_API_KEY[:20]}..." if OPENROUTER_API_KEY else "❌ No OPENROUTER API key")
print(f"✓ COHERE API Key loaded: {COHERE_API_KEY[:20]}..." if COHERE_API_KEY else "❌ No COHERE API key")

# %%
# Import ابزارها و utils
from legal_multi_agent.utils.toon import extract_toon_verifier
from legal_multi_agent.tools.option_verifier_tool import (
    option_verifier_tool,
    verify_options_direct
)
from legal_multi_agent.tools.retriever_tool import (
    retriever_tool,
    retrieve_documents
)

print("✓ Imports successful")

# %% [markdown]
## 2️⃣ تست TOON Parser با نمونه ساده

# %%
sample_toon_1 = """
بررسی گزینه‌ها:

results{option,support_level,reasoning}:
1,NOT_SUPPORTED,طبق ماده 826 قانون مدنی این گزینه صحیح نیست
2,SUPPORTED,ماده 416 قانون مدنی حق فسخ و ارش را تایید می‌کند
3,UNCLEAR,شواهد کافی برای تایید یا رد این گزینه وجود ندارد
4,NOT_SUPPORTED,مغایر با ماده 795 قانون مدنی

results{recommended_answer,confidence}:
2,5

توضیحات اضافی...
"""

result_1 = extract_toon_verifier(sample_toon_1, verbose=True)
print("\n" + "="*60)
print("نتیجه Parse:")
print("="*60)

if result_1:
    print(f"✓ تعداد امتیازها: {len(result_1['scores'])}")
    print(f"✓ گزینه پیشنهادی: {result_1['recommended_answer']}")
    print(f"✓ اطمینان: {result_1['confidence']}/5")
    print("\nجزئیات امتیازها:")
    for score in result_1['scores']:
        print(f"  گزینه {score['option_number']}: {score['support_level']}")
        print(f"    دلیل: {score['reasoning'][:60]}...")
else:
    print("❌ Parse ناموفق بود")

# %% [markdown]
## 3️⃣ تست با نمونه TOON ناقص (edge case)

# %%
# تست وقتی فقط 3 گزینه امتیاز دارد
sample_toon_incomplete = """
results{option,support_level,reasoning}:
1,SUPPORTED,گزینه اول درست است
2,NOT_SUPPORTED,گزینه دوم غلط است
3,UNCLEAR,نمی‌دانم

results{recommended_answer,confidence}:
1,3
"""

result_incomplete = extract_toon_verifier(sample_toon_incomplete, verbose=True)
print("\n" + "="*60)
print("نتیجه Parse (ناقص):")
print("="*60)
if result_incomplete:
    print(f"✓ تعداد امتیازها: {len(result_incomplete['scores'])}")
    print(f"✓ گزینه پیشنهادی: {result_incomplete['recommended_answer']}")
else:
    print("❌ Parse ناموفق")

# %% [markdown]
## 4️⃣ تست Option Verifier Tool با یک سوال ساده

# %%
# یک سوال ساده حقوقی
test_question = "اگر مبیع معیوب باشد، خریدار چه حقی دارد؟"

test_options = """1) فقط حق خیار دارد
2) حق فسخ یا مطالبه ارش دارد
3) هیچ حقی ندارد
4) فقط می‌تواند ارش بگیرد"""

test_sources = """
ماده 416 قانون مدنی:
هرگاه مبیع معیوب باشد، خریدار می‌تواند معامله را فسخ کند یا مبیع را با ارش نگه دارد.

ماده 417 قانون مدنی:
در صورتی که بایع تضمین سلامت مبیع را نکرده باشد، خریدار نمی‌تواند خسارت مطالبه کند.

ماده 826 قانون مدنی:
عیب چیزی است که موجب کاهش قیمت یا ارزش مبیع شود.
"""

print("="*60)
print("تست Option Verifier Tool")
print("="*60)
print(f"\nسوال: {test_question}")
print(f"\nگزینه‌ها:\n{test_options}")
print(f"\nمنابع موجود: {len(test_sources)} کاراکتر")

# %%
# فراخوانی tool
print("\n⏳ در حال فراخوانی option_verifier_tool...")

result_str = option_verifier_tool.invoke({
    "question": test_question,
    "options_text": test_options,
    "sources": test_sources,
})

print("\n" + "="*60)
print("خروجی Tool:")
print("="*60)
print(result_str)

# %% [markdown]
## 5️⃣ تست verify_options_direct (نسخه بدون wrapper)

# %%
print("\n" + "="*60)
print("تست verify_options_direct")
print("="*60)

result_dict = verify_options_direct(
    question=test_question,
    options_text=test_options,
    sources=test_sources,
)

print(f"\nنوع خروجی: {type(result_dict)}")
print(f"کلیدها: {list(result_dict.keys())}")

if "scores" in result_dict and result_dict["scores"]:
    print(f"\n✓ تعداد امتیازها: {len(result_dict['scores'])}")
    print(f"✓ گزینه پیشنهادی: {result_dict.get('recommended_answer')}")
    print(f"✓ اطمینان: {result_dict.get('confidence')}/5")
    
    print("\nجزئیات:")
    for score in result_dict['scores']:
        print(f"\nگزینه {score['option_number']}: {score['support_level']}")
        print(f"  {score['reasoning']}")
else:
    print("❌ خطا در دریافت نتیجه:")
    print(result_dict.get("error", "نامشخص"))

# %% [markdown]
## 6️⃣ تست با سوال پیچیده‌تر (ارث)

# %%
complex_question = """مردی فوت می‌کند و وارث او همسر، پدر و یک پسر است. 
سهم همسر از ترکه چقدر است؟"""

complex_options = """1) یک هشتم
2) یک چهارم
3) یک ششم
4) نصف"""

complex_sources = """
ماده 913 قانون مدنی:
نصیب زوج از ترکه زوجه که فرزند ندارد نصف است و اگر فرزند داشته باشد ربع.

ماده 914 قانون مدنی:
نصیب زوجه از ترکه زوج که فرزند ندارد ربع است و اگر فرزند داشته باشد ثمن.

ماده 875 قانون مدنی:
سهم پدر در حضور فرزند پسر یک ششم است.

ماده 864 قانون مدنی:
فرزند پسر نسبت به سایر وراث مقدم است.
"""

print("="*60)
print("تست سوال پیچیده (ارث)")
print("="*60)
print(f"\nسوال: {complex_question}")

# %%
print("\n⏳ در حال تحلیل...")

complex_result = verify_options_direct(
    question=complex_question,
    options_text=complex_options,
    sources=complex_sources,
)

if complex_result.get("scores"):
    print("\n✓ تحلیل موفق")
    print(f"گزینه پیشنهادی: {complex_result['recommended_answer']}")
    print(f"اطمینان: {complex_result['confidence']}/5")
    
    print("\nتحلیل هر گزینه:")
    for score in complex_result['scores']:
        status_emoji = {
            "SUPPORTED": "✅",
            "NOT_SUPPORTED": "❌",
            "UNCLEAR": "❓"
        }.get(score['support_level'], "•")
        
        print(f"\n{status_emoji} گزینه {score['option_number']}: {score['support_level']}")
        print(f"   {score['reasoning']}")
else:
    print("❌ خطا:")
    print(complex_result.get("error"))
    if "raw" in complex_result:
        print("\nخروجی خام:")
        print(complex_result["raw"])

# %% [markdown]
## 7️⃣ بررسی کیفیت استدلال‌ها

# %%
def analyze_reasoning_quality(result_dict):
    """تحلیل کیفیت استدلال‌های تولید شده"""
    if not result_dict.get("scores"):
        return "❌ بدون نتیجه"
    
    scores = result_dict["scores"]
    
    print("تحلیل کیفیت استدلال‌ها:")
    print("="*60)
    
    # چک کردن ارجاع به مواد
    article_citations = 0
    for score in scores:
        reasoning = score["reasoning"]
        # جستجو برای "ماده" یا عدد (مثلاً "416")
        if "ماده" in reasoning or any(str(num) in reasoning for num in range(1, 1500)):
            article_citations += 1
    
    print(f"✓ تعداد استدلال‌های دارای ارجاع به ماده: {article_citations}/{len(scores)}")
    
    # توزیع support levels
    levels = [s["support_level"] for s in scores]
    from collections import Counter
    level_counts = Counter(levels)
    
    print(f"\nتوزیع امتیازها:")
    for level, count in level_counts.items():
        print(f"  {level}: {count}")
    
    # طول متوسط استدلال
    avg_length = sum(len(s["reasoning"]) for s in scores) / len(scores)
    print(f"\nطول متوسط استدلال: {avg_length:.0f} کاراکتر")
    
    # اطمینان
    confidence = result_dict.get("confidence")
    if confidence:
        print(f"\nسطح اطمینان کلی: {confidence}/5")
        if confidence >= 4:
            print("  → اطمینان بالا ✓")
        elif confidence >= 3:
            print("  → اطمینان متوسط")
        else:
            print("  → اطمینان پایین ⚠️")

# تحلیل نتیجه سوال ساده
print("\n📊 تحلیل سوال ساده:")
if result_dict.get("scores"):
    analyze_reasoning_quality(result_dict)

# تحلیل نتیجه سوال پیچیده
print("\n📊 تحلیل سوال پیچیده:")
if complex_result.get("scores"):
    analyze_reasoning_quality(complex_result)

# %% [markdown]
## 8️⃣ تست Retriever Tool با فیلترها

# %%
print("="*60)
print("تست Retriever Tool با فیلترها")
print("="*60)

# تست 1: بدون فیلتر
print("\n1️⃣ جستجوی عمومی:")
result1 = retriever_tool.invoke({
    "query": "حق خیار عیب",
    "top_k": 3,
})
print(result1[:800])

# تست 2: با فیلتر ماده
print("\n\n2️⃣ فیلتر روی ماده 416:")
result2 = retriever_tool.invoke({
    "query": "عیب مبیع",
    "top_k": 2,
    "article_number": "416",
})
print(result2[:800])

# تست 3: با فیلتر قانون
print("\n\n3️⃣ فیلتر روی قانون مدنی:")
result3 = retriever_tool.invoke({
    "query": "خیار",
    "top_k": 3,
    "law_name": "قانون مدنی",
})
print(result3[:800])

# %% [markdown]
## 9️⃣ تست retrieve_documents (نسخه مستقیم)

# %%
print("\n" + "="*60)
print("تست retrieve_documents (برای استفاده در nodes)")
print("="*60)

docs_result = retrieve_documents(
    query="حق فسخ قرارداد",
    top_k=3,
    use_rerank=True,
)

print(f"\nنوع خروجی: {type(docs_result)}")
print(f"کلیدها: {list(docs_result.keys())}")
print(f"\n✓ تعداد نتایج: {len(docs_result['rag_results'])}")
print(f"✓ طول context: {len(docs_result['context'])} کاراکتر")
print(f"✓ تعداد metadata: {len(docs_result['docs_meta'])}")

print("\nمتادیتای اسناد:")
for meta in docs_result['docs_meta'][:3]:
    print(f"  [{meta['i']}] {meta['source_type']} - {meta['law']} - ماده {meta.get('article_number', 'N/A')}")

print("\nPreview context:")
print(docs_result['context_preview'][:500])

# %% [markdown]
## 🔟 خلاصه نتایج

# %%
print("="*60)
print("خلاصه تست‌ها")
print("="*60)

tests_summary = {
    "TOON Parser (نمونه کامل)": result_1 is not None,
    "TOON Parser (نمونه ناقص)": result_incomplete is not None,
    "Option Verifier Tool": "گزینه پیشنهادی" in result_str,
    "verify_options_direct": result_dict.get("scores") is not None,
    "سوال پیچیده": complex_result.get("scores") is not None,
    "Retriever Tool (عمومی)": "تعداد اسناد" in result1,
    "Retriever Tool (فیلتر ماده)": "ماده" in result2 or "هیچ سندی" in result2,
    "Retriever Tool (فیلتر قانون)": "قانون" in result3 or "تعداد" in result3,
    "retrieve_documents": docs_result.get("rag_results") is not None,
}

for test_name, passed in tests_summary.items():
    status = "✅ موفق" if passed else "❌ ناموفق"
    print(f"{status} - {test_name}")

# محاسبه درصد موفقیت
success_rate = sum(tests_summary.values()) / len(tests_summary) * 100
print(f"\n📊 نرخ موفقیت کلی: {success_rate:.0f}%")

if success_rate == 100:
    print("\n🎉 همه تست‌ها موفق بودند! ابزارها آماده استفاده هستند.")
elif success_rate >= 80:
    print("\n✓ اکثر تست‌ها موفق بودند. ابزارها قابل استفاده هستند.")
else:
    print("\n⚠️ برخی تست‌ها ناموفق بودند. نیاز به بررسی دارد.")

# %%


✓ Project root: f:\Thesis\project\3-Multi-Agent-System
✓ Src path: f:\Thesis\project\3-Multi-Agent-System\src
✓ OPENROUTER API Key loaded: sk-or-v1-10d2ef641ff...
✓ COHERE API Key loaded: rq8LfpFUvkclAvSyxHfD...
✓ Imports successful

نتیجه Parse:
✓ تعداد امتیازها: 4
✓ گزینه پیشنهادی: 2
✓ اطمینان: 5/5

جزئیات امتیازها:
  گزینه 1: NOT_SUPPORTED
    دلیل: طبق ماده 826 قانون مدنی این گزینه صحیح نیست...
  گزینه 2: SUPPORTED
    دلیل: ماده 416 قانون مدنی حق فسخ و ارش را تایید می‌کند...
  گزینه 3: UNCLEAR
    دلیل: شواهد کافی برای تایید یا رد این گزینه وجود ندارد...
  گزینه 4: NOT_SUPPORTED
    دلیل: مغایر با ماده 795 قانون مدنی...

نتیجه Parse (ناقص):
✓ تعداد امتیازها: 3
✓ گزینه پیشنهادی: 1
تست Option Verifier Tool

سوال: اگر مبیع معیوب باشد، خریدار چه حقی دارد؟

گزینه‌ها:
1) فقط حق خیار دارد
2) حق فسخ یا مطالبه ارش دارد
3) هیچ حقی ندارد
4) فقط می‌تواند ارش بگیرد

منابع موجود: 284 کاراکتر

⏳ در حال فراخوانی option_verifier_tool...

خروجی Tool:
✓ نتیجه تحلیل گزینه‌ها:
  گزینه 1: NOT_SUPPORTED

In [ ]:
# %%
print("="*60)
print("تست مستقیم: آیا ماده 416 در دیتابیس هست؟")
print("="*60)

# تست 1: بدون فیلتر، کوئری دقیق
result_direct = retriever_tool.invoke({
    "query": "ماده 416 قانون مدنی مبیع معیوب",
    "top_k": 5,
})
print("\n1️⃣ بدون فیلتر:")
print(result_direct[:600])

# تست 2: با فیلتر ماده
result_filtered = retriever_tool.invoke({
    "query": "ماده 416",
    "top_k": 5,
    "article_number": "416",
})
print("\n\n2️⃣ با فیلتر:")
print(result_filtered[:600])

تست مستقیم: آیا ماده 416 در دیتابیس هست؟

1️⃣ بدون فیلتر:
✓ تعداد اسناد بازیابی شده: 5
✓ Rerank فعال: True

📄 خلاصه اسناد:
[1] قانون | قانون: قانون مدنی | ماده 416
[2] قانون | قانون: قانون مدنی | ماده 422
[3] قانون | قانون: قانون مدنی | ماده 434
[4] دادنامه | قانون: قانون آیین دادرسی مدنی | قانون مدنی | عنوان: درخواست تایید فسخ قرارداد پیش از اعمال آ...
[5] وحدت رویه | قانون: قانون مدنی | عنوان: مسئولیت بایع در جبران کاهش ارزش ثمن در ب...

متن کامل اسناد:

[منبع 1] (قانون) - قانون مدنی
شماره_ماده: 416
article_number: 416
volume: ج


2️⃣ با فیلتر:
✓ تعداد اسناد بازیابی شده: 5
✓ فیلتر ماده: 416
✓ Rerank فعال: True

📄 خلاصه اسناد:
[1] قانون | قانون: قانون مدنی | ماده 416
[2] قانون | قانون: قانون آیین دادرسی دادگاه‌های عمومی و انقلاب در امور مدنی | ماده 416
[3] قانون | قانون: قانون تجارت | ماده 416
[4] قانون | قانون: قانون مجازات اسلامی مصوب 1392 | ماده 416
[5] وحدت رویه | قانون: قانون مدنی | عنوان: شرط سقوط خیار غبن و تأثیر آن بر مراتب اف...

متن کامل اسناد:

[منبع 1] (قانون) - قانون مدنی